## **Исследование активности пользователей на Hacker News: какие типы постов получают больше комментариев (часть 1)**

В этом проекте мы будем исследовать посты на популярном сайте Hacker News, где пользователи публикуют новости и обсуждения, связанные с технологиями. Главная цель - выяснить, какие типы постов получают больше комментариев: посты с вопросами к сообществу (Ask HN) или посты с демонстрацией интересных проектов и открытий (Show HN).

Кроме того, мы изучим, как время создания поста влияет на количество комментариев, чтобы понять, есть ли оптимальное время для публикации, которое способствует более активному обсуждению.

Откроем наш набор данных `'hacker_news.csv'` и сохраним его в виде списка списков (list of lists):

In [1]:
from csv import reader

with open('hacker_news.csv') as file:
    read_file = reader(file)
    news_data = list(read_file)

#Сохраним заголовки столбцов в отдельную переменную
headers = news_data[0]

# Перезапишем наш набор данных уже без строки с заголовками
news_data = news_data[1:]

print(f"Заголовки столбцов:\n{headers}\n")
print(f"Первые 5 записей:\n{news_data[:5]}")

Заголовки столбцов:
['id', 'title', 'url', 'num_points', 'num_comments', 'author', 'created_at']

Первые 5 записей:
[['12224879', 'Interactive Dynamic Video', 'http://www.interactivedynamicvideo.com/', '386', '52', 'ne0phyte', '8/4/2016 11:52'], ['10975351', 'How to Use Open Source and Shut the Fuck Up at the Same Time', 'http://hueniverse.com/2016/01/26/how-to-use-open-source-and-shut-the-fuck-up-at-the-same-time/', '39', '10', 'josep2', '1/26/2016 19:30'], ['11964716', "Florida DJs May Face Felony for April Fools' Water Joke", 'http://www.thewire.com/entertainment/2013/04/florida-djs-april-fools-water-joke/63798/', '2', '1', 'vezycash', '6/23/2016 22:20'], ['11919867', 'Technology ventures: From Idea to Enterprise', 'https://www.amazon.com/Technology-Ventures-Enterprise-Thomas-Byers/dp/0073523429', '3', '1', 'hswarna', '6/17/2016 0:01'], ['10301696', 'Note by Note: The Making of Steinway L1037 (2007)', 'http://www.nytimes.com/2007/11/07/movies/07stein.html?_r=0', '8', '2', 'walterbel

---

Создадим отдельные списки для каждого из трёх типов постов:

1. \#Ask Hacker News
2. \#Show Hacker News
3. Other posts

In [2]:
ask_posts = []
show_posts = []
other_posts = []

for row in news_data:
    title = row[1].lower()
    
    if title.startswith('ask hn'):
        ask_posts.append(row)
    elif title.startswith('show hn'):
        show_posts.append(row)
    else:
        other_posts.append(row)
        
print(f"The number of posts in the ask_posts: {len(ask_posts)}")
print(f"The number of posts in the show_posts: {len(show_posts)}")
print(f"The number of posts in the other_posts: {len(other_posts)}")

The number of posts in the ask_posts: 1744
The number of posts in the show_posts: 1162
The number of posts in the other_posts: 17194


---

Исследуем, какие типы постов в среднем получают наибольшее число комментариев (Ask HN или Show HN). Для этого создадим функцию `calculate_avg_comments()`. Она принимает два параметра:

- data - список списков, где каждый вложенный список представляет собой данные одного поста;

- column_index - индекс столбца, в котором находится количество комментариев.

Функция вычисляет и возвращает среднее количество комментариев для всех постов из переданного списка.

In [3]:
def calculate_avg_comments(data, column_index):
    total_comments = 0
    for row in data:
        total_comments += int(row[column_index])  
        
    return round(total_comments / len(data))

In [4]:
avg_ask_comments = calculate_avg_comments(ask_posts, 4)
print(f"Average number of comments for Ask HN posts: {avg_ask_comments}\n")

avg_show_comments = calculate_avg_comments(show_posts, 4)
print(f"Average number of comments for Show HN posts: {avg_show_comments}\n")

Average number of comments for Ask HN posts: 14

Average number of comments for Show HN posts: 10



Как видно из результатов, посты с вопросами к сообществу (Ask HN) в среднем получают больше комментариев, чем посты с публикацией проектов (Show HN).

Это может быть связано с тем, что людям интереснее общаться, делиться мнением и вступать в дискуссии, нежели просто оценивать чужие работы. Однако стоит отметить, что разница в среднем числе комментариев между двумя типами постов не слишком велика - примерно 14 для Ask HN и 10 для Show HN.

---

Поскольку посты с вопросами чаще получают комментарии, мы сосредоточим наш оставшийся анализ именно на них. Теперь нам предстоит выяснить, влияет ли время публикации поста на количество оставленных комментариев.

In [5]:
import datetime as dt

# Список, где каждая запись содержит дату публикации и число комментариев для постов Ask HN
result_list = []

for row in ask_posts:
    num_comments = int(row[4])
    publication_date = row[6]
    result_list.append([publication_date, num_comments])
    
    
# Словарь, где ключ - час суток (от 0 до 23), а значение - список из двух чисел:
# [количество постов, сумма комментариев], опубликованных в этот час
posts_comments_by_hour_dict = {}


# Проходим циклом по каждой записи из result_list и обновляем значения в словаре posts_comments_by_hour_dict
for row in result_list:
    dt_object = dt.datetime.strptime(row[0], "%m/%d/%Y %H:%M")
    publication_time = dt_object.strftime("%H") #Извлекаем из даты только час создания поста
    num_comments = row[1]
    
    if publication_time not in posts_comments_by_hour_dict:
        posts_comments_by_hour_dict[publication_time] = [1, num_comments] 
    else:
        posts_comments_by_hour_dict[publication_time][0] += 1
        posts_comments_by_hour_dict[publication_time][1] += num_comments
        

        
# Список, где каждая запись содержит час публикации и среднее количество комментариев для этого часа
avg_by_hour_list = []

for key, value in posts_comments_by_hour_dict.items():
    count, comments = value
    avg_by_hour_list.append([key, round(comments / count)])
    
avg_by_hour_list = sorted(avg_by_hour_list)

Проанализируем среднее количество комментариев для каждого часа:

In [6]:
for row in avg_by_hour_list:
    print(f"Publication hour '{row[0]}:00' has average number of comments: {row[1]}")

Publication hour '00:00' has average number of comments: 8
Publication hour '01:00' has average number of comments: 11
Publication hour '02:00' has average number of comments: 24
Publication hour '03:00' has average number of comments: 8
Publication hour '04:00' has average number of comments: 7
Publication hour '05:00' has average number of comments: 10
Publication hour '06:00' has average number of comments: 9
Publication hour '07:00' has average number of comments: 8
Publication hour '08:00' has average number of comments: 10
Publication hour '09:00' has average number of comments: 6
Publication hour '10:00' has average number of comments: 13
Publication hour '11:00' has average number of comments: 11
Publication hour '12:00' has average number of comments: 9
Publication hour '13:00' has average number of comments: 15
Publication hour '14:00' has average number of comments: 13
Publication hour '15:00' has average number of comments: 39
Publication hour '16:00' has average number of 

Выведем топ-5 часов для публикации поста, которые потенциально могут привести к наибольшему числу комментариев:

In [7]:
# Новый список, где в каждой записи из avg_by_hour_list элементы поменяны местами:
# теперь первый элемент - среднее количество комментариев, второй - час публикации
swap_avg_by_hour_list = []

for row in avg_by_hour_list:
    swap_avg_by_hour_list.append([row[1], row[0]])

# Сортируем список по среднему количеству комментариев в порядке убывания
swap_avg_by_hour_list = sorted(swap_avg_by_hour_list, reverse = True)

# Выведем топ-5 часов с наибольшим средним количеством комментариев
for row in swap_avg_by_hour_list[:5]:
    print(f"'{row[1]}:00': {row[0]} average comments per post.")

'15:00': 39 average comments per post.
'02:00': 24 average comments per post.
'20:00': 22 average comments per post.
'16:00': 17 average comments per post.
'21:00': 16 average comments per post.


### **Выводы:**

Как видно из результатов, наибольшее число комментариев получают посты, публикуемые в `02:00`, `15:00-16:00` и `20:00-21:00`. Несмотря на то, что это общее количество комментариев за всё время существования постов, видно явное деление пользователей Hacker News на «ночных» и «дневных».

Можно предположить, что часть пользователей - «ночные совы», которые предпочитают работать или бодрствовать ночью. Возможно, они работают из дома или в ночных заведениях, а в свободное время заходят на сайт, чтобы поделиться своим мнением относительно заданного вопроса.

Вторая группа - «дневные» пользователи, придерживающиеся стандартного рабочего графика. Пик активности в `15:00-16:00`, вероятно, связан с посещением форума во время рабочего перерыва. Повышенная активность около `20:00-21:00` может объясняться возвращением пользователей домой после работы, в том числе во время поездки и ожидания в пробках.

В остальных временных диапазонах заметной активности пользователей не наблюдается.

---
---

## **Сравнение среднего количества баллов `Points` для постов Ask HN и Show HN (часть 2)**

Ранее было установлено, что посты Ask HN в среднем получают больше комментариев. В рамках данного анализа сравнивается среднее количество баллов `points` для постов Ask HN и Show HN. Показатель `points` отражает итоговую оценку публикации сообществом и рассчитывается как разница между количеством положительных и отрицательных оценок. Сравнение средних значений позволит определить, какой тип контента получает более высокую оценку пользователей Hacker News.

Создадим функцию `calculate_avg_points()`. Она принимает два параметра:

- data - список списков, где каждый вложенный список содержит данные одного поста;

- column_index - индекс столбца с количеством баллов (`points`). 

Функция вычисляет и возвращает среднее количество оценок для всех постов из переданного списка.

In [8]:
def calculate_avg_points(data, column_index):
    total_points = 0
    for row in data:
        total_points += int(row[column_index]) # Суммируем баллы из каждого поста  
        
    avg_points = total_points / len(data)  # Вычисляем среднее значение
    return round(avg_points, 2) 

In [9]:
avg_ask_points = calculate_avg_points(ask_posts, 3)
print(f"Average number of points for Ask HN posts: {avg_ask_points}\n")

avg_show_points = calculate_avg_points(show_posts, 3)
print(f"Average number of points for Show HN posts: {avg_show_points}\n")

Average number of points for Ask HN posts: 15.06

Average number of points for Show HN posts: 27.56



### **Выводы:**

Анализ показал, что посты Ask HN в среднем получают больше комментариев (`14` против `10`), тогда как посты Show HN набирают значительно больше баллов (`27,56` против `15,06`).

- Можно предположить, что более низкий средний показатель `points` у постов Ask HN может быть связан с большим количеством отрицательных оценок по сравнению с Show HN. Однако показатель `points` представляет собой итоговую разницу между положительными и отрицательными оценками, поэтому на основе имеющихся данных невозможно достоверно определить, связано ли это различие с количеством отрицательных оценок, меньшим числом положительных или сочетанием обоих факторов.

Таким образом, посты Ask HN генерируют больше обсуждений, тогда как Show HN получают более высокую оценку со стороны сообщества. Вероятно, это связано с различиями в целях публикаций: Ask HN ориентированы на получение ответов и обмен мнениями, что способствует более активному обсуждению и большему количеству комментариев. В свою очередь, Show HN обычно представляют готовый проект, продукт или результат работы автора, что может вызывать положительную реакцию сообщества и выражаться в более высоком количестве баллов. При этом пользователи могут ограничиваться оценкой публикации без участия в обсуждении. Кроме того, если автор поста явно не запрашивает обратную связь, пожелания или рекомендации, мотивация оставлять комментарии может быть ниже.